# A LoKI sessionThe local application of the architecture sketch is a notebook on the clientinterface (D9). This one tells one session on LoKI@Larmor: find the beam centre,pick a background run, reduce to I(Q), change the Q binning interactively, forkthe plot, and read the provenance back.Every step below is the sketch's vocabulary: a **record** is a request plus whathappened to it, a **reference** is how a request names data, a **label** is theslot an interactive tool owns, the **picker** is the query behind an input field,and a **stage** is the part of the sciline graph a warm workflow holds.

## The client`local` puts client, backend, launcher, session, and data store in this process.The specs are bound in-process, which local mode allows as long as no installedpackage claims the same name and version. The `FolderSource` is the datasetsource (D7): the folder esssans caches the tutorial files in. Their names are`<run>-<date>.nxs`, so the source is told the shape of the run identity and theinstrument the names do not carry.

In [ ]:
import tempfile
import time
from pathlib import Path

import ipywidgets as widgets
import plopp as pp
from IPython.display import display

from ess.apps import loki
from ess.apps.client import local
from ess.apps.sources import FolderSource
from ess.apps.spec import DatasetRef

cache = loki.cache()
client = local(
    Path(tempfile.mkdtemp(prefix='loki-session-')),
    instrument='loki',
    proposal='p1',
    submitter='notebook',
    registry=loki.registry(),
    sources=[FolderSource(cache, identity=r'(?P<run>\d+)-.*', instrument='loki')],
)
cache

## The picker`client.pick()` is what an input field asks: the datasets of every source and theoutputs of completed records, as rows of one shape. Nothing is stored to make thelist. A file whose name carries a run number is identified by instrument and run,which is what a PID is minted from; the direct-beam file carries none, so itspath is its identity.

In [ ]:
for candidate in client.pick():
    print(f'{candidate.ref!s:<28} {candidate.display["name"]}')

In [ ]:
# Pick the background run by its run number, and the rest of the LoKI@Larmor set.
background = DatasetRef(instrument='loki', run=60393)
inputs = {
    'sample_run': DatasetRef(instrument='loki', run=60387),          # AgBeh
    'sample_transmission_run': DatasetRef(instrument='loki', run=60386),
    'background_run': background,
    'background_transmission_run': DatasetRef(instrument='loki', run=60392),
    'empty_beam_run': DatasetRef(instrument='loki', run=60392),
    'direct_beam': DatasetRef(path=cache / 'direct-beam-loki-all-pixels.h5'),
}
background

## A record: the beam centreThe beam centre is its own spec because it is its own run: one sample run in, onesmall value out. The output is a `Quantity`, a vocabulary value, so it is storedin the record itself rather than in the data store.

In [ ]:
center = client.run(loki.BEAM_CENTER, {'sample_run': inputs['sample_run']})
print(center.id, center.status.value, center.spec)
print('resolved params:', center.resolved_params)
print('output:', center.outputs['center'])

## A reference: I(Q) from the beam centre`center.ref()` is "output `center` of record `center.id`". It goes into therequest as a reference and stays there, which is what makes provenance the graphyou get by following references. The `beam_center` parameter is a union of aliteral and a reference, so a user may equally well type a vector in.

In [ ]:
def request(**overrides):
    return inputs | {'beam_center': center.ref(), 'q_bins': 100} | overrides


started = time.perf_counter()
iofq = client.run(loki.IOFQ, request(), label='iofq')
print(f'{iofq.status.value} in {time.perf_counter() - started:.2f} s, '
      f'reused={iofq.reused}')
pp.plot(client.output(iofq, 'iofq'), norm='log')

## A slot: the Q binning on a slider`q_min`, `q_max` and `q_bins` are declared cheap on the spec, so the bindingbuilds a `sciline.Stage` whose inputs they are. Everything the Q binning cannotaffect -- loading, masking, the wavelength and Q conversions, thetransmission and direct-beam normalisation -- is held at the stage's frontier,and a rerun only histograms and subtracts.Each slider move submits a complete request under the label `iofq`. That label isthe slot the plot owns: the newest record under it supersedes the earlier ones,and none of them is lost.`record.reused` says only that the runner kept the callable between runs, notthat the stage was reused; the time per rerun is what shows that.

In [ ]:
log = []
slider = widgets.IntSlider(value=100, min=25, max=300, step=25, description='Q bins')


def rebin(change):
    started = time.perf_counter()
    record = client.run(loki.IOFQ, request(q_bins=change['new']), label='iofq')
    log.append(
        f'q_bins={change["new"]:>3}  {record.id}  reused={record.reused}  '
        f'{time.perf_counter() - started:.2f} s'
    )
    display(pp.plot(client.output(record, 'iofq'), norm='log'))


slider.observe(rebin, names='value')
slider

In [ ]:
# Executed headless, these stand in for dragging the slider.
for value in (200, 50, 300):
    slider.value = value
print('\n'.join(log))

## The slot's records`client.latest` answers the one query the backend offers for a label. `batch` isthe latest record per member key, which for a slider with no member key is thatsame record; the full history under the label is an ordinary record query.

In [ ]:
print('latest:', client.latest('iofq').id)
print('batch: ', [r.id for r in client.batch('iofq')])
for record in client.records(label='iofq'):
    print(record.id, record.resolved_params['q_bins'], record.status.value)

## Forking the slotComparing two variants side by side is two labels. The second is assigned whenthe user forks; discarding a variant drops its label from the UI and nothingelse.

In [ ]:
fine = client.run(loki.IOFQ, request(q_bins=300), label='iofq-fine')
coarse = client.latest('iofq')
display(widgets.HBox([
    pp.plot(client.output(coarse, 'iofq'), norm='log', title='iofq').to_widget(),
    pp.plot(client.output(fine, 'iofq'), norm='log', title='iofq-fine').to_widget(),
]))

## ProvenanceFollowing the references of the latest I(Q) record reaches the beam-centre recordand, through both, the dataset references the requests name. Datasets are theleaves: they have no record and nothing to recompute.

In [ ]:
provenance = client.provenance(client.latest('iofq'))
print(provenance['spec'], provenance['package_versions'], provenance['binding'])
print('datasets:', [str(DatasetRef(**raw)) for raw in provenance['raw']])
for upstream in provenance['inputs']:
    print('from', upstream['spec'], upstream['record'],
          'over', [str(DatasetRef(**raw)) for raw in upstream['raw']])